In [8]:
#import module for dibetes dieseas prediction.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib

In [9]:
#data preprocessing.
df=pd.read_csv("diabetes.csv")
df.head(5)
df.isnull().sum()
zero=(df==0).sum()
print(zero)

Pregnancies                 111
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64


In [10]:
cols=['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI']
for col in cols:
    df[col]=df[col].replace(0,df[col].mean())

In [11]:
zero=(df==0).sum()
print(zero)

Pregnancies                   0
Glucose                       0
BloodPressure                 0
SkinThickness                 0
Insulin                       0
BMI                           0
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64


In [12]:
#split the data forn train and test

X=df.drop("Outcome",axis=1)
Y=df['Outcome']

x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42)


In [13]:
#scale feature from the dataset
scale=StandardScaler()

x_train_scale=scale.fit_transform(x_train)
x_test_scale=scale.transform(x_test)

In [14]:
sm=SMOTE(random_state=42)
x_train_res,y_train_res=sm.fit_resample(x_train_scale,y_train)

In [15]:
#train the model
model=RandomForestClassifier(n_estimators=100,class_weight='balanced', random_state=42)

model.fit(x_train_res,y_train_res)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [16]:
base_model=[
    ('rf',RandomForestClassifier(n_estimators=100,class_weight='balanced', random_state=42)),
    ('gb',GradientBoostingClassifier(n_estimators=100,learning_rate=0.1,random_state=42))
]

In [17]:
meta_model=LogisticRegression(max_iter=1000,random_state=42)

In [18]:
Stacking=StackingClassifier(
    estimators=base_model,
    final_estimator=meta_model,
    cv=5,
    stack_method='predict_proba',
    n_jobs=-1
)

In [19]:
Stacking.fit(x_train_res,y_train_res)

StackingClassifier(cv=5,
                   estimators=[('rf',
                                RandomForestClassifier(class_weight='balanced',
                                                       random_state=42)),
                               ('gb',
                                GradientBoostingClassifier(random_state=42))],
                   final_estimator=LogisticRegression(max_iter=1000,
                                                      random_state=42),
                   n_jobs=-1, stack_method='predict_proba')

In [20]:
y_pred1=Stacking.predict(x_test_scale)
y_prob1=Stacking.predict_proba(x_test_scale)[:,1]

In [21]:
y_pred=model.predict(x_test_scale)

In [22]:
print(y_pred)

[1 0 0 0 1 1 0 1 1 1 0 1 0 0 0 1 0 0 1 1 0 0 0 0 1 1 0 0 0 0 1 1 1 1 1 1 1
 0 0 1 0 0 1 1 0 1 1 0 0 1 0 1 1 0 0 0 1 0 0 1 1 0 0 0 0 1 0 1 0 1 1 0 0 0
 0 1 0 0 0 0 1 0 0 1 0 1 1 1 0 0 0 0 0 1 0 1 1 0 1 0 1 0 1 1 1 0 0 1 0 1 0
 0 0 1 0 1 1 1 0 0 0 0 1 0 0 0 1 1 1 1 1 1 0 0 1 0 0 1 1 0 0 0 0 0 0 0 0 0
 0 1 0 0 1 0]


In [23]:
y_prob=model.predict_proba(x_test_scale)[:,1]


In [24]:
print(y_prob)

[0.55 0.23 0.12 0.25 0.61 0.62 0.02 0.81 0.73 0.71 0.23 0.73 0.37 0.23
 0.03 0.51 0.13 0.02 0.72 0.67 0.41 0.06 0.27 0.07 0.73 0.9  0.15 0.01
 0.15 0.24 0.72 0.8  0.75 0.91 0.63 0.85 0.84 0.41 0.21 0.78 0.06 0.32
 0.8  0.57 0.04 0.72 0.68 0.33 0.15 0.93 0.02 0.89 0.83 0.26 0.11 0.03
 0.75 0.09 0.27 0.83 0.76 0.32 0.29 0.37 0.07 0.79 0.04 0.53 0.03 0.74
 0.8  0.12 0.08 0.02 0.17 0.53 0.21 0.12 0.2  0.26 0.83 0.1  0.1  0.74
 0.42 0.92 0.55 0.67 0.4  0.06 0.01 0.18 0.09 0.61 0.44 0.65 0.77 0.1
 0.69 0.06 0.64 0.05 0.67 0.8  0.9  0.33 0.29 0.95 0.18 0.85 0.03 0.41
 0.3  0.79 0.27 0.53 0.86 0.63 0.03 0.43 0.   0.14 0.54 0.05 0.24 0.36
 0.62 0.85 0.67 0.64 0.54 0.72 0.03 0.49 0.7  0.34 0.21 0.64 0.82 0.08
 0.   0.   0.29 0.46 0.1  0.32 0.23 0.02 0.39 0.86 0.14 0.33 0.57 0.09]


In [25]:
#evalution of model

print(f"the accuracy of the model:{accuracy_score(y_test,y_pred)}")
print(f"the accuracy of the model:{accuracy_score(y_test,y_pred1)}")


the accuracy of the model:0.7727272727272727
the accuracy of the model:0.7792207792207793


In [26]:
print("the confusion matix:\n")
print(confusion_matrix(y_test,y_pred1))


the confusion matix:

[[78 21]
 [13 42]]


In [27]:
print("the classification matrix:\n")
print(classification_report(y_test,y_pred1))

the classification matrix:

              precision    recall  f1-score   support

           0       0.86      0.79      0.82        99
           1       0.67      0.76      0.71        55

    accuracy                           0.78       154
   macro avg       0.76      0.78      0.77       154
weighted avg       0.79      0.78      0.78       154



In [28]:
joblib.dump(Stacking,'diabetes.pkl')
joblib.dump(scale,'scale.pkl')

['scale.pkl']